# Generating the synthetic data 

In [1]:
using DifferentialEquations
using DataFrames
using CSV

Algo to generate the synthetic data
step 1 :  define the constants 
step 2 : define all the funtion required 
step 3 : create a funtion for differential eq. (for   4d ode)



### The defined constants

In [2]:

# 1. Numerically Stable Rate Equations using expm1 to handle 0/0 smoothly
function alpha_m(V)
    # Standard: 0.1*(V + 40) / (1 - exp(-(V + 40)/10))
    # Rewritten using expm1 to avoid 0/0 singularity at V = -40
    x = -(V + 40.0) / 10.0
    return (x == 0.0) ? 1.0 : (0.1 * (V + 40.0)) / (-expm1(x))
end

function beta_m(V)
    return 4.0 * exp(-(V + 65.0) / 18.0)
end

function alpha_h(V)
    return 0.07 * exp(-(V + 65.0) / 20.0)
end

function beta_h(V)
    return 1.0 / (1.0 + exp(-(V + 35.0) / 10.0))
end

function alpha_n(V)
    # Standard: 0.01*(V + 55) / (1 - exp(-(V + 55)/10))
    # Rewritten using expm1 to avoid 0/0 singularity at V = -55
    x = -(V + 55.0) / 10.0
    return (x == 0.0) ? 0.1 : (0.01 * (V + 55.0)) / (-expm1(x))
end

function beta_n(V)
    return 0.125 * exp(-(V + 65.0) / 80.0)
end





beta_n (generic function with 1 method)

In [3]:

# 2. Define the 4D Hodgkin-Huxley Vector Field
function hodgkin_huxley!(du, u, p, t)
    # Explicit state tracking: u = [V, m, h, n]
    V, m, h, n = u
    Cm, g_Na, g_K, g_L, E_Na, E_K, E_L = p
    
    # Stimulus current: 10 uA/cm^2 applied between t=1.0ms and t=3.0ms
    # This window guarantees a clean, stable upstroke and subsequent refractory period
    I_ext = (2.0 <= t <= 30.0) ? 10.0 : 0.0
    
    # Calculate Ionic Currents
    I_Na = g_Na * (m^3) * h * (V - E_Na)
    I_K  = g_K * (n^4) * (V - E_K)
    I_L  = g_L * (V - E_L)
    
    # Derivatives mapping exactly to state indexes
    du[1] = (I_ext - I_Na - I_K - I_L) / Cm  # dV/dt
    du[2] = alpha_m(V) * (1.0 - m) - beta_m(V) * m  # dm/dt
    du[3] = alpha_h(V) * (1.0 - h) - beta_h(V) * h  # dh/dt
    du[4] = alpha_n(V) * (1.0 - n) - beta_n(V) * n  # dn/dt
end

# 3. Execution Script Setup
# Analytical resting equilibria values matching [V, m, h, n]
u0 = [-65.0, 0.0529, 0.5961, 0.3177] 
tspan = (0.0, 30.0) # Expanded window to observe full action potential and recovery

# Standard Biophysical Parameters 
# [Cm, g_Na, g_K, g_L, E_Na, E_K, E_L]
p = [1.0, 120.0, 36.0, 0.3, 50.0, -77.0, -54.4]


7-element Vector{Float64}:
   1.0
 120.0
  36.0
   0.3
  50.0
 -77.0
 -54.4

In [4]:
# Setup the ODE problem
prob = ODEProblem(hodgkin_huxley!, u0, tspan, p)

# Solve using high-order stiff Radau solver with dense sampling (0.01 ms steps)
println("Solving 4D Hodgkin-Huxley equations via RadauIIA5 solver...")
sol = solve(prob, RadauIIA5(), saveat=0.01, reltol=1e-8, abstol=1e-8)

# 4. Construct DataFrame ensuring flawless structural mapping
df = DataFrame(
    t = sol.t,
    V = [u[1] for u in sol.u],
    m = [u[2] for u in sol.u],
    h = [u[3] for u in sol.u],  # Explicitly mapping index 3 to h
    n = [u[4] for u in sol.u]   # Explicitly mapping index 4 to n
)

Solving 4D Hodgkin-Huxley equations via RadauIIA5 solver...


Row,t,V,m,h,n
,Float64,Float64,Float64,Float64,Float64
1,0.0,-65.0,0.0529,0.5961,0.3177
2,0.01,-65.0,0.0529013,0.5961,0.3177
3,0.02,-65.0001,0.0529026,0.5961,0.3177
4,0.03,-65.0001,0.0529038,0.5961,0.3177
5,0.04,-65.0001,0.052905,0.5961,0.3177
6,0.05,-65.0001,0.0529061,0.5961,0.3177
7,0.06,-65.0002,0.0529071,0.5961,0.3177
8,0.07,-65.0002,0.0529081,0.5961,0.3177
9,0.08,-65.0002,0.052909,0.5961,0.3177


In [5]:


# Preview data structure
println("Data Generation Complete! Sample DataFrame structure:")
println(first(df, 10))

# Export clean CSV file
CSV.write("HH_ground_truth_profile_A.csv", df)
println("Saved safely to 'HH_ground_truth_profile_A.csv'. Ready for PyTorch data ingestion.")


Data Generation Complete! Sample DataFrame structure:
10×5 DataFrame
 Row │ t        V         m          h        n       
     │ Float64  Float64   Float64    Float64  Float64 
─────┼────────────────────────────────────────────────
   1 │    0.0   -65.0     0.0529      0.5961   0.3177
   2 │    0.01  -65.0     0.0529013   0.5961   0.3177
   3 │    0.02  -65.0001  0.0529026   0.5961   0.3177
   4 │    0.03  -65.0001  0.0529038   0.5961   0.3177
   5 │    0.04  -65.0001  0.052905    0.5961   0.3177
   6 │    0.05  -65.0001  0.0529061   0.5961   0.3177
   7 │    0.06  -65.0002  0.0529071   0.5961   0.3177
   8 │    0.07  -65.0002  0.0529081   0.5961   0.3177
   9 │    0.08  -65.0002  0.052909    0.5961   0.3177
  10 │    0.09  -65.0002  0.05291     0.5961   0.3177
Saved safely to 'HH_ground_truth_profile_A.csv'. Ready for PyTorch data ingestion.


In [6]:
import Pkg
Pkg.add("Plots")

    Updating registry at `C:\Users\ADMIN\.julia\registries\General.toml`
   Resolving package versions...
     Project No packages added to or removed from `C:\Users\ADMIN\.julia\environments\v1.12\Project.toml`
    Manifest No packages added to or removed from `C:\Users\ADMIN\.julia\environments\v1.12\Manifest.toml`


In [ ]:
using Plots

# Create a layout with 2 subplots vertically stacked
p1 = plot(df.t, df.V, label="Vm (Membrane Potential)", color=:crimson, linewidth=2, ylabel="Voltage (mV)")
title!("4D Hodgkin-Huxley Trajectory")

p2 = plot(df.t, df.m, label="m (Na activation)", color=:blue, linewidth=1.5)
plot!(df.t, df.h, label="h (Na inactivation)", color=:green, linewidth=1.5)
plot!(df.t, df.n, label="n (K activation)", color=:orange, linewidth=1.5, ylabel="Gate Probability", xlabel="Time (ms)")

# Combine into a single layout sharing the X-axis
plot(p1, p2, layout=(2,1), size=(800,600), link=:x)
